# State and Gate Teleportation

This notebook builds the two teleportation primitives that Clifford Noise Reduction (CliNR) [1, 2] rests on. We start with quantum state teleportation, which moves an unknown single-qubit state from one qubit to another using a shared Bell pair, a measurement, and a Pauli correction. We then extend it to gate teleportation, which applies a Clifford operation by building it into the resource state ahead of time, so that consuming the resource applies the operation and leaves only a Pauli correction.

These two ideas are all we need to understand CliNR, which is developed in the companion notebook *Clifford Noise Reduction (CliNR)*. Here we focus on getting the primitives right and verifying each one against its definition in simulation.

### What you will learn

1. How state teleportation moves a quantum state using a Bell pair, a measurement, and a correction.
2. How gate teleportation applies a Clifford operation by building it into the resource state, leaving only a Pauli correction.
3. How to verify each protocol exactly in simulation with [Stim](https://github.com/quantumlib/Stim).

## Setup

In [ ]:
# If these are not already available in your environment:
# %pip install stim numpy

import stim
import numpy as np

## 1. State teleportation

State teleportation transfers an unknown single-qubit state from one qubit to another using a shared Bell pair and two bits of classical communication.

We use three qubits:

- qubit 0 holds the state to be teleported, $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$,
- qubits 1 and 2 hold a Bell pair, $|B_{00}\rangle = \tfrac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.

The initial joint state is $|\psi\rangle_0 \otimes |B_{00}\rangle_{1,2}$, with the subscripts naming the qubits each factor lives on.

To see how teleportation works, we regroup this *same* state around the pair we are about to measure, qubits 0 and 1, expanding them in the Bell basis. Writing $|B_{ab}\rangle_{0,1}$ for the Bell states of qubits 0 and 1:

$$
\begin{aligned}
|\psi\rangle_0 \, |B_{00}\rangle_{1,2} = \tfrac{1}{2}\Big(
  & |B_{00}\rangle_{0,1}\, (\alpha|0\rangle + \beta|1\rangle)_2
  + |B_{01}\rangle_{0,1}\, (\alpha|1\rangle + \beta|0\rangle)_2 \\
  &+ |B_{10}\rangle_{0,1}\, (\alpha|0\rangle - \beta|1\rangle)_2
  + |B_{11}\rangle_{0,1}\, (\alpha|1\rangle - \beta|0\rangle)_2
\Big).
\end{aligned}
$$

Nothing about the state has changed, only the grouping: $|B_{00}\rangle_{1,2}$ on the left describes qubits 1 and 2, while each $|B_{ab}\rangle_{0,1}$ on the right describes qubits 0 and 1 — the pair the protocol measures.

Measuring qubits 0 and 1 in the Bell basis therefore collapses qubit 2 into one of four states, each a known Pauli applied to $|\psi\rangle$. Labeling the two measurement outcomes $a$ (from qubit 0) and $b$ (from qubit 1), qubit 2 holds $Z^{a} X^{b} |\psi\rangle$. Applying the correction $X^{b}$ followed by $Z^{a}$ recovers $|\psi\rangle$ exactly.

In the circuit, the Bell-basis measurement is implemented with a CNOT followed by a Hadamard, after which qubits 0 and 1 are measured in the computational basis.

In [ ]:
def build_state_teleportation():
    """
    State teleportation of qubit 0 onto qubit 2 using a Bell pair on qubits 1 and 2.
    The correction on qubit 2 is applied in-circuit via classical feedforward.
    """
    c = stim.Circuit()

    # Prepare a nontrivial state on qubit 0 to teleport (H then S, for illustration).
    c.append("H", [0])
    c.append("S", [0])

    # Create the Bell pair |B_00> on qubits 1 and 2.
    c.append("H", [1])
    c.append("CX", [1, 2])

    # Bell-basis measurement of qubits 0 and 1: CNOT then Hadamard, then measure.
    c.append("CX", [0, 1])
    c.append("H", [0])
    c.append("M", [0])          # outcome a  -> recorded as rec[-2] after next line
    c.append("M", [1])          # outcome b  -> recorded as rec[-1]

    # Feedforward correction on qubit 2: X^b then Z^a.
    c.append("CX", [stim.target_rec(-1), 2])   # X conditioned on b (qubit 1 outcome)
    c.append("CZ", [stim.target_rec(-2), 2])   # Z conditioned on a (qubit 0 outcome)

    return c

state_teleport = build_state_teleportation()

In [ ]:
# Render the circuit as an SVG diagram
# White panel keeps the black gate glyphs legible on any notebook theme
# (stim draws them black-on-transparent, which vanishes on a dark background).
from IPython.display import HTML
HTML(f'<div style="background:white; padding:10px; overflow-x:auto">'
     f'{state_teleport.diagram("timeline-svg")}</div>')

### Verifying it works

To confirm teleportation succeeded, we check the protocol against its definition: whatever single-qubit state we prepare on qubit 0 should appear on qubit 2 after the correction. A direct way to test this is to verify that the same preparation, applied to qubit 0, and then "undone" on qubit 2, returns qubit 2 to $|0\rangle$ on every shot.

We build a small test circuit: prepare the state, teleport it, then apply the inverse preparation to qubit 2 and measure. A correct protocol yields outcome 0 on qubit 2 in every shot.

In [ ]:
def test_state_teleportation(shots=2000, seed=0):
    """
    Teleport H S |0> from qubit 0 to qubit 2, then apply (H S)^dagger = S^dag H
    to qubit 2 and measure it. A correct protocol gives outcome 0 every time.
    """
    # Build the teleportation circuit
    c = build_state_teleportation() 

    # Undo the preparation on qubit 2: inverse of (H then S) is (S_DAG then H).
    c.append("S_DAG", [2])
    c.append("H", [2])

    # final measurement of qubit 2 to check if it is back to |0>
    c.append("M", [2]) 
                         
    samples = c.compile_sampler(seed=seed).sample(shots)
    qubit2_outcomes = samples[:, -1]        
    return int(qubit2_outcomes.sum())

nonzero = test_state_teleportation()
print(f"Shots where qubit 2 was not |0> after teleport-and-undo: {nonzero}")
assert nonzero == 0, "State teleportation failed: qubit 2 was not restored to |0>."
print("State teleportation verified: the input state was recovered on qubit 2.")

## 2. Gate teleportation

Instead of teleporting a state unchanged, we build a Clifford operation $C$ into the resource state ahead of time. Teleportation then delivers $C|\psi\rangle$ rather than $|\psi\rangle$.

In the example below, we illustrate with a two-qubit Clifford $C$ acting on the teleported register.

### Why it works

Start from the same Bell pair, but apply $C$ to the second half of it. The resource state becomes $(I \otimes C)|B_{00}\rangle$. Running the teleportation protocol with this resource teleports $|\psi\rangle$ through $C$: the output qubits hold $C|\psi\rangle$, again up to a Pauli byproduct set by the measurement outcomes.

Concretely, the circuit uses six qubits: two **data** qubits (0 and 1) holding $|\psi\rangle$, and two sets of **resource** qubits, `res_a` (2 and 3) and `res_b` (4 and 5). We prepare $|\psi\rangle$ on the data qubits, a Bell pair on each of $(q2, q4)$ and $(q3, q5)$, then apply $C$ to the `res_b` half (qubits 4 and 5). We follow with a Bell-basis measurement of each data qubit with its `res_a` partner — $(q0, q2)$ and $(q1, q3)$.

In plain state teleportation the output carries the byproduct $Z^{a}X^{b}$ (Section 1). Gate teleportation runs that protocol with $C$ built into the resource, so the output register (`res_b`, qubits 4 and 5) ends up holding $C|\psi\rangle$ dressed by a Pauli byproduct. 

Writing $o_0, o_1, o_2, o_3$, with $o_i \in \{0,1\}$, for the measurement outcomes of qubits 0, 1, 2, 3, each data outcome ($o_0, o_1$) drives a $Z$ and each resource-partner outcome ($o_2, o_3$) drives an $X$ on the matching output qubit. Because $C$ is Clifford, conjugating that byproduct by $C$ keeps it a Pauli, and the correction that recovers $C|\psi\rangle$ is

$$Q = \big(C^{\dagger} X_4 C\big)^{o_2}\big(C^{\dagger} Z_4 C\big)^{o_0}\,\big(C^{\dagger} X_5 C\big)^{o_3}\big(C^{\dagger} Z_5 C\big)^{o_1}.$$

In [ ]:
def small_clifford_on(c, q0, q1):
    """
    A small illustrative two-qubit Clifford C applied to qubits (q0, q1):
    Hadamard on q0, then CNOT from q0 to q1. Used as the gate to teleport.
    """
    c.append("H", [q0])
    c.append("CX", [q0, q1])

def build_gate_teleportation():
    """
    Gate teleportation of a two-qubit Clifford C.
    Data register: qubits 0, 1 (state |psi>).
    Resource: two Bell pairs on (2,4) and (3,5); C is applied to the (4,5) half.
    Output C|psi> appears on qubits 4, 5 after the feedforward Pauli correction.
    """
    c = stim.Circuit()
    data = [0, 1]
    res_a = [2, 3]      # first set of resource qubits: halves measured with the data
    res_b = [4, 5]      # second set of resource qubits: carry C, hold the output

    # Prepare a nontrivial input state |psi> on the data qubits.
    c.append("H", [0])
    c.append("S", [1])

    # Two Bell pairs on the two sets of resource qubits: (2,4) and (3,5).
    for a, b in zip(res_a, res_b):
        c.append("H", [a])
        c.append("CX", [a, b])

    # Build C into the resource by applying it to the res_b half.
    small_clifford_on(c, res_b[0], res_b[1])

    # Teleport: Bell-basis measurement of each data qubit with its res_a partner.
    for d, a in zip(data, res_a):
        c.append("CX", [d, a])
        c.append("H", [d])
    c.append("M", data)     # data outcomes  -> drive the Z-byproduct correction
    c.append("M", res_a)    # res_a outcomes -> drive the X-byproduct correction

    # Feedforward Pauli correction on the output register res_b. The teleported state
    # carries a Pauli byproduct; conjugating it by C (forward, C P C^dag) gives the Pauli
    # that cancels it, leaving C|psi>. We obtain those Paulis by embedding C's tableau on
    # res_b and pushing X and Z through it -- the same construction the CliNR builder uses.
    c_only = stim.Circuit(); small_clifford_on(c_only, 0, 1)
    full = stim.Tableau(6)                                   # identity on all six qubits
    full.append(stim.Tableau.from_circuit(c_only), res_b)    # embed C onto the res_b qubits
    for i in range(2):
        xi = stim.PauliString(6); xi[res_b[i]] = 1           # X on output qubit i
        zi = stim.PauliString(6); zi[res_b[i]] = 3           # Z on output qubit i
        x_corr, z_corr = full(xi), full(zi)                  # C X C^dag and C Z C^dag on res_b
        # Record order is [data0, data1, res_a0, res_a1]: the data outcome for pair i sits at
        # rec[-4+i] and conditions the Z correction; the res_a outcome at rec[-2+i] conditions X.
        for pauli, rec in ((z_corr, stim.target_rec(-4 + i)),
                           (x_corr, stim.target_rec(-2 + i))):
            xs, zs = pauli.to_numpy()
            for q in range(6):
                if   xs[q] and zs[q]: c.append("CY", [rec, q])
                elif xs[q]:           c.append("CX", [rec, q])
                elif zs[q]:           c.append("CZ", [rec, q])

    return c, data, res_a, res_b

gate_teleport, data, res_a, res_b = build_gate_teleportation()

In [ ]:
# The full protocol: resource preparation, C, teleportation measurements, and the
# feedforward correction on the output register (qubits 4, 5).
# White panel keeps the black gate glyphs legible on any notebook theme
# (stim draws them black-on-transparent, which vanishes on a dark background).
from IPython.display import HTML
HTML(f'<div style="background:white; padding:10px; overflow-x:auto">'
     f'{gate_teleport.diagram("timeline-svg")}</div>')

### Verifying it works

As with state teleportation, we check the protocol against its definition: the output register should hold $C|\psi\rangle$. We prepare the input, run the full gate teleportation including the feedforward correction, then apply the inverse of "prepare $|\psi\rangle$, then apply $C$" to the output qubits. A correct protocol returns those qubits to $|0\rangle$ on every shot, and because the correction depends on the random measurement outcomes, passing across many shots exercises it for every outcome combination.

In [ ]:
def test_gate_teleportation(shots=2000, seed=0):
    """
    Run gate teleportation, then undo it on the output register and measure. The output is
    C|psi>, so applying (prepare |psi>, then C)^dagger returns res_b to |00> on every shot
    whenever the feedforward correction is right -- for every measurement outcome.
    """
    c, data, res_a, res_b = build_gate_teleportation()

    # Rebuild "prepare |psi>, then apply C" on the output qubits res_b, and append its inverse.
    undo = stim.Circuit()
    undo.append("H", [res_b[0]]); undo.append("S", [res_b[1]])   # the input preparation
    small_clifford_on(undo, res_b[0], res_b[1])                  # C
    c += undo.inverse()

    c.append("M", res_b)                     # measure the two output qubits
    samples = c.compile_sampler(seed=seed).sample(shots)
    return int(samples[:, -2:].sum())        # last two columns are res_b; expect 0 everywhere

nonzero = test_gate_teleportation()
print(f"Shots where the output register was not |00> after teleport-and-undo: {nonzero}")
assert nonzero == 0, "Gate teleportation failed: the output register was not C|psi>."
print("Gate teleportation verified: the output register holds C|psi>.")

## What's next?

We learnt how gate teleportation lets a quantum operation be applied on the data qubits by preparing it on a resource state, consuming the resource, and finally, correcting a Pauli byproduct at the end. 

The next notebook, *Notebook 2 - Clifford Noise Reduction (CliNR)*, adds one ingredient to this picture — verifying the resource state before it is consumed — and shows how that turns teleportation into a noise reduction mechanism!

## References

[1] N. Delfosse and E. Tham, "Clifford Noise Reduction," arXiv:2407.06583 (2024).

[2] E. Tham and N. Delfosse, "Optimized Clifford Noise Reduction: Theory, Simulations and Experiments," *Quantum* **9**, 1829 (2025). [arXiv:2504.13356](https://arxiv.org/abs/2504.13356)